# ⚡ Day 01a: Python Async/Await Deep Dive

---

## 🎯 What You'll Master Today
- How async/await works under the hood
- The event loop explained
- asyncio core: gather(), wait(), create_task()
- Sync vs async performance comparison
- Common async patterns and pitfalls

---

## 🎭 The Restaurant Analogy

**Sync waiter:** Takes order, goes to kitchen, WAITS until food is ready, delivers, then takes next order. One customer at a time.

**Async waiter:** Takes order, sends to kitchen, immediately takes next order. When food is ready, delivers it. Multiple customers at once.

The waiter is **one thread** — but they serve many customers because they don't **block** on waiting.

```
╔═══════════════════════════════════════════════════════════════════╗
║  ASYNC/AWAIT — CORE CONCEPTS                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  async def  → declares a coroutine (not a regular function)     ║
║  await      → yields control to event loop ("I'm waiting")      ║
║  event loop → scheduler that runs coroutines cooperatively      ║
║  coroutine  → function that can pause and resume                ║
║                                                                   ║
║  KEY INSIGHT:                                                    ║
║  async is for I/O-bound tasks (network, disk, DB)               ║
║  threading/multiprocessing for CPU-bound tasks                   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Coroutines — The Foundation
# ============================================================
import asyncio
import time

# A coroutine is defined with async def
async def greet(name: str) -> str:
    print(f"  Hello, {name}! (about to await)")
    await asyncio.sleep(1)  # Simulates I/O — yields control
    print(f"  Done waiting, {name}!")
    return f"Greeting for {name}"

# Calling a coroutine doesn't run it — it returns a coroutine object
coro = greet("Sarthak")
print(f"Type: {type(coro)}")

# You must await it or pass it to the event loop
result = await coro
print(f"Result: {result}")

In [ ]:
# ============================================================
# Sync vs Async — Performance Comparison
# ============================================================

# SYNC: Sequential execution
def sync_fetch(url_id: int) -> str:
    time.sleep(1)  # Simulates 1 second I/O
    return f"Result-{url_id}"

start = time.perf_counter()
results = [sync_fetch(i) for i in range(5)]
sync_time = time.perf_counter() - start
print(f"Sync:  {sync_time:.2f}s for 5 calls (sequential)")

# ASYNC: Concurrent execution
async def async_fetch(url_id: int) -> str:
    await asyncio.sleep(1)  # Simulates 1 second I/O
    return f"Result-{url_id}"

start = time.perf_counter()
results = await asyncio.gather(
    async_fetch(0), async_fetch(1), async_fetch(2),
    async_fetch(3), async_fetch(4)
)
async_time = time.perf_counter() - start
print(f"Async: {async_time:.2f}s for 5 calls (concurrent)")
print(f"\nSpeedup: {sync_time/async_time:.1f}x faster!")

In [ ]:
# ============================================================
# asyncio.gather() — Run Multiple Coroutines Concurrently
# ============================================================

async def fetch_user(user_id: int):
    await asyncio.sleep(0.5)
    return {"id": user_id, "name": f"User-{user_id}"}

async def fetch_posts(user_id: int):
    await asyncio.sleep(0.3)
    return [{"title": f"Post by {user_id}"}]

async def fetch_notifications(user_id: int):
    await asyncio.sleep(0.4)
    return ["Notif 1", "Notif 2"]

# Run all three concurrently — total time = max(0.5, 0.3, 0.4) ≈ 0.5s
start = time.perf_counter()
user, posts, notifs = await asyncio.gather(
    fetch_user(1),
    fetch_posts(1),
    fetch_notifications(1)
)
elapsed = time.perf_counter() - start

print(f"User:   {user}")
print(f"Posts:  {posts}")
print(f"Notifs: {notifs}")
print(f"Total:  {elapsed:.2f}s (not 1.2s!)")

In [ ]:
# ============================================================
# create_task() — Fire-and-Forget vs Await
# ============================================================

async def background_work(task_name: str, seconds: float):
    print(f"  [{task_name}] Starting...")
    await asyncio.sleep(seconds)
    print(f"  [{task_name}] Done after {seconds}s")
    return f"{task_name} result"

# create_task() schedules the coroutine to run but doesn't wait
task1 = asyncio.create_task(background_work("Task-A", 0.5))
task2 = asyncio.create_task(background_work("Task-B", 0.3))

print("Tasks created, doing other work...")
await asyncio.sleep(0.1)
print("...other work done, now waiting for tasks")

# Await the tasks to get results
result1 = await task1
result2 = await task2
print(f"Results: {result1}, {result2}")

In [ ]:
# ============================================================
# asyncio.wait() — More Control Than gather()
# ============================================================

async def flaky_task(task_id: int):
    await asyncio.sleep(0.2 * task_id)
    if task_id == 3:
        raise ValueError(f"Task {task_id} failed!")
    return f"Task-{task_id} done"

tasks = [asyncio.create_task(flaky_task(i)) for i in range(1, 5)]

# wait() gives you done and pending sets
done, pending = await asyncio.wait(tasks, return_when=asyncio.ALL_COMPLETED)

for task in done:
    if task.exception():
        print(f"❌ Failed: {task.exception()}")
    else:
        print(f"✅ {task.result()}")

print("\n# gather() vs wait():")
print("#   gather() — returns results in order, raises on first exception")
print("#   wait()   — returns done/pending sets, handles exceptions per-task")

In [ ]:
# ============================================================
# Timeouts — Don't Wait Forever
# ============================================================

async def slow_operation():
    await asyncio.sleep(10)
    return "Done"

# Method 1: asyncio.wait_for()
try:
    result = await asyncio.wait_for(slow_operation(), timeout=1.0)
except asyncio.TimeoutError:
    print("❌ wait_for: Timed out after 1s")

# Method 2: asyncio.timeout() (Python 3.11+)
try:
    async with asyncio.timeout(1.0):
        result = await slow_operation()
except asyncio.TimeoutError:
    print("❌ timeout ctx: Timed out after 1s")

print("\n💡 Always set timeouts on external calls!")

In [ ]:
# ============================================================
# Semaphores — Limit Concurrency
# ============================================================

# Without semaphore: 1000 concurrent DB connections → crash!
# With semaphore: max 10 concurrent connections

semaphore = asyncio.Semaphore(3)  # Max 3 concurrent

async def limited_fetch(task_id: int):
    async with semaphore:
        print(f"  Task {task_id} running")
        await asyncio.sleep(0.5)
        print(f"  Task {task_id} done")
        return task_id

start = time.perf_counter()
results = await asyncio.gather(*[limited_fetch(i) for i in range(9)])
elapsed = time.perf_counter() - start

print(f"\n9 tasks with semaphore(3): {elapsed:.2f}s")
print("Expected: ~1.5s (3 batches × 0.5s each)")

In [ ]:
# ============================================================
# Common Pitfalls
# ============================================================

print("""
PITFALL 1: Forgetting to await
─────────────────────────────
❌ result = async_function()    → Returns coroutine, doesn't execute!
✅ result = await async_function()

PITFALL 2: Blocking the event loop
──────────────────────────────────
❌ time.sleep(5)                → Blocks entire event loop!
✅ await asyncio.sleep(5)       → Yields control

❌ requests.get(url)            → Blocks event loop!
✅ async with httpx.AsyncClient() as c: await c.get(url)

PITFALL 3: CPU-bound in async
─────────────────────────────
❌ async def process(): heavy_computation()  → Blocks loop
✅ await asyncio.to_thread(heavy_computation) → Runs in thread pool
✅ Use multiprocessing for true CPU parallelism

PITFALL 4: Not handling exceptions in tasks
────────────────────────────────────────────
❌ asyncio.create_task(might_fail())  → Exception silently lost!
✅ task = asyncio.create_task(might_fail())
   try:
       await task
   except Exception:
       handle error
""")

In [ ]:
# ============================================================
# Real-World Pattern: Async API Client
# ============================================================

print("""
Production async pattern — httpx + gather + semaphore:

import httpx

MAX_CONCURRENT = 10
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

async def fetch_url(client: httpx.AsyncClient, url: str):
    async with semaphore:
        try:
            response = await asyncio.wait_for(
                client.get(url), timeout=10.0
            )
            return {"url": url, "status": response.status_code}
        except asyncio.TimeoutError:
            return {"url": url, "error": "timeout"}
        except Exception as e:
            return {"url": url, "error": str(e)}

async def fetch_all(urls: list[str]):
    async with httpx.AsyncClient() as client:
        tasks = [fetch_url(client, url) for url in urls]
        return await asyncio.gather(*tasks)

# Usage:
# results = await fetch_all(["https://api1.com", "https://api2.com", ...])
""")

print("Key patterns:")
print("  1. Shared httpx.AsyncClient (connection pooling)")
print("  2. Semaphore for concurrency limits")
print("  3. Timeout on every external call")
print("  4. Per-task error handling")

---

## 🗺️ Async Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  ASYNC/AWAIT DECISION GUIDE                                      ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  TASK TYPE:                                                      ║
║    I/O-bound  → async/await (network, disk, DB)                 ║
║    CPU-bound  → multiprocessing or asyncio.to_thread()          ║
║    Mixed      → async + run_in_executor() for CPU parts         ║
║                                                                   ║
║  CONCURRENCY TOOLS:                                              ║
║    gather()      → run N coroutines, get results in order       ║
║    wait()        → get done/pending, handle errors per-task     ║
║    create_task() → schedule without waiting, await later        ║
║    Semaphore     → limit max concurrent (rate limiting)         ║
║    wait_for()    → timeout on single coroutine                  ║
║                                                                   ║
║  LIBRARIES:                                                      ║
║    HTTP:  httpx (async) — NOT requests                          ║
║    DB:    asyncpg, aiomysql, motor (MongoDB)                    ║
║    Files: aiofiles                                               ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | How does async/await work? | Coroutines yield control at await points; event loop schedules them cooperatively |
| 2 | Async vs threading? | Async = cooperative, single thread, I/O-bound. Threading = preemptive, multiple threads, GIL limited |
| 3 | What is the event loop? | Scheduler that runs coroutines, checks I/O readiness, manages callbacks |
| 4 | gather() vs wait()? | gather: ordered results, raises on error. wait: done/pending sets, per-task error handling |
| 5 | When NOT to use async? | CPU-bound tasks, simple scripts, when sync is fast enough |
| 6 | How to limit concurrency? | asyncio.Semaphore — limits how many coroutines run simultaneously |
| 7 | What blocks the event loop? | time.sleep(), requests.get(), CPU computation, any sync I/O call |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • async def creates a coroutine, await pauses it       │
## │  • Event loop schedules coroutines cooperatively         │
## │  • gather() for concurrent I/O (5 calls in 1s, not 5s) │
## │  • Semaphore limits concurrency (don't DDoS yourself)   │
## │  • Never use blocking calls inside async functions       │
## │  • Always set timeouts on external calls                 │
## │  • Use httpx (not requests) in async code               │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Celery & Redis** — Task queues for heavy background processing